# BART: Transformer-based Text Summarization

In [2]:
from tqdm import tqdm
from transformers import BartTokenizer, BartForConditionalGeneration

In [4]:
# Load model + tokenizer
model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 14546.46it/s]


## Save the BART Model Locally

In [6]:
import os
from pathlib import Path

In [8]:
CWD         = os.getcwd()
MODEL_DIR   = "models"
MODEL_PATH  = os.path.join(CWD, MODEL_DIR)

os.makedirs(name=MODEL_PATH, exist_ok=True)

In [9]:
# save locally
save_dir    = "bart_model"
save_path   = os.path.join(MODEL_PATH, save_dir)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.95s/it]


('/home/cyrus/UTS-OneDrive/2026/42850 NLP Algorithms/Assignments/Assignment_2/NLP_Team5_Assignment3/models/bart_model/tokenizer_config.json',
 '/home/cyrus/UTS-OneDrive/2026/42850 NLP Algorithms/Assignments/Assignment_2/NLP_Team5_Assignment3/models/bart_model/tokenizer.json')

In [10]:
# Reload the locally saved model
tokenizer   = BartTokenizer.from_pretrained(save_path)
model       = BartForConditionalGeneration.from_pretrained(save_path)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 5287.45it/s]


In [11]:
text = """
Netflix users may terminate their subscription at any time.
However, refunds are only provided within 30 days of purchase.
The company reserves the right to modify these terms without notice.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Summary:")
print(summary)

Summary:
Netflix users may terminate their subscription at any time. Refunds are only provided within 30 days of purchase. Company reserves the right to modify these terms without notice.


## Load the Dataset

In [17]:
import json

In [13]:
CWD         = os.getcwd()
DATA_PATH   = Path(CWD) / "data" / "all_v1.json"

In [20]:
with open(DATA_PATH, 'r') as f:
    json_str = f.read()
    data = json.loads(json_str)

In [27]:
# View the first 5 texts and their BART summaries
for i, (key, value) in enumerate(data.items()):
    print(key)
    print("Document:", value['doc'])
    print("Title:", value['title'])
    print("Original text:", value['original_text'])
    print("Summary text:", value['reference_summary'])

    text = value['original_text']

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    start_color = '\033[91m'    # red
    end_color   = '\033[0m'     # white

    print(f"{start_color}BART Summary: {summary}{end_color}")
    print()
    if i > 5:
        break

legalsum01
Document: Pokemon GO Terms of Service
Title: 
Original text: welcome to the pokémon go video game services which are accessible via the niantic inc niantic mobile device application the app. to make these pokémon go terms of service the terms easier to read our video game services the app and our websites located at http pokemongo nianticlabs com and http www pokemongolive com the site are collectively called the services. please read carefully these terms our trainer guidelines and our privacy policy because they govern your use of our services.
Summary text: hi.
BART Summary: Our video game services the app and our websites located at http pokemongo nianticlabs com and http www pokemongolive com the site are collectively called the services. Please read carefully these terms our trainer guidelines and our privacy policy because they govern your use of our services.

legalsum02
Document: Pokemon GO Terms of Service
Title: Agreement To Terms
Original text: by using our servi